# 03. Modelo de Priorización

## Objetivo

Construir un modelo de priorización para identificar clientes con mayor probabilidad de activar una tarjeta física en una siguiente ola de entrega, considerando que la disponibilidad de tarjetas es limitada.

La priorización se plantea como un problema de propensión: estimar la probabilidad de activación observada de una tarjeta física a partir de información disponible antes de su emisión.

Para este ejercicio se asume que `fecha_activacion` representa el proceso de activación utilizado por peiGo, el cual requiere una acción explícita del cliente posterior a la emisión.

### Definición del target

- `target = 1`: la tarjeta física registra activación dentro de los 30 días posteriores a su emisión.
- `target = 0`: no registra activación dentro de dicha ventana.

La ventana de 30 días se utiliza porque todas las activaciones observadas en el piloto ocurrieron dentro de los primeros 24 días posteriores a la emisión.

Idealmente, la respuesta podría complementarse con el uso posterior de la tarjeta física. Sin embargo, las transacciones no contienen una llave que permita asociarlas con una `tarjeta_id` específica, por lo que no es posible distinguir de forma confiable si una transacción posterior fue realizada con la tarjeta física o virtual.

> **Limitación:** El modelo estima probabilidad de activación observada bajo el proceso representado en los datos; no una medida pura de intención del cliente ni del efecto incremental de entregarle una tarjeta física.


## 0. Configuración

In [59]:
# Librerías y configuración

from io import BytesIO

import boto3
import joblib
import numpy as np
import pandas as pd

from debit_card_pilot.config import (
    ACTIVATION_WINDOW_DAYS,
    CONTROL_INDEX_DATE,
    GOLD_PREFIX,
    MODEL_ARTIFACT_PREFIX,
    PRE_WINDOW_DAYS,
    S3_BUCKET,
    SILVER_PREFIX,
)

AWS_PROFILE = "ds-technical-test"

session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

In [60]:
# Lectura de datasets Silver
# Lee todos los archivos Parquet de un prefijo de S3 y los consolida en un DataFrame.

def read_s3_parquet_folder(folder: str) -> pd.DataFrame:
    prefix = f"{SILVER_PREFIX}/{folder}/"

    paginator = s3.get_paginator("list_objects_v2")
    parquet_files = []

    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        parquet_files.extend(
            obj["Key"]
            for obj in page.get("Contents", [])
            if obj["Key"].endswith(".parquet")
        )

    dataframes = []

    for key in parquet_files:
        response = s3.get_object(Bucket=S3_BUCKET, Key=key)
        dataframes.append(
            pd.read_parquet(BytesIO(response["Body"].read()))
        )

    return pd.concat(dataframes, ignore_index=True)

In [61]:
# Carga de Silver
# Recupera las fuentes necesarias para la construcción del modelo de priorización.

customers = read_s3_parquet_folder("customers")
cards = read_s3_parquet_folder("cards")
transactions = read_s3_parquet_folder("transactions")

## 1. Definición de Población y Target

In [62]:
# Clasificación de tenencia de tarjetas
# Identifica clientes con tarjeta física, virtual o ambas.

card_ownership = (
    cards
    .assign(
        has_physical=cards["tipo"].eq("fisica"),
        has_virtual=cards["tipo"].eq("virtual"),
    )
    .groupby("cliente_id", as_index=False)
    .agg(
        has_physical=("has_physical", "max"),
        has_virtual=("has_virtual", "max"),
    )
)

card_ownership["card_group"] = "other"

card_ownership.loc[
    card_ownership["has_physical"] & card_ownership["has_virtual"],
    "card_group",
] = "physical_and_virtual"

card_ownership.loc[
    card_ownership["has_physical"] & ~card_ownership["has_virtual"],
    "card_group",
] = "physical_only"

card_ownership.loc[
    ~card_ownership["has_physical"] & card_ownership["has_virtual"],
    "card_group",
] = "virtual_only"

display(card_ownership["card_group"].value_counts())

card_group
virtual_only            6309
physical_and_virtual    4894
physical_only            488
Name: count, dtype: int64

In [63]:
# Poblaciones de modelado
# Entrena con clientes que tenían virtual y posteriormente recibieron física.

valid_customer_ids = set(customers["cliente_id"])

train_customer_ids = set(
    card_ownership.loc[
        card_ownership["card_group"].eq("physical_and_virtual"),
        "cliente_id",
    ]
) & valid_customer_ids

candidate_customer_ids = set(
    card_ownership.loc[
        card_ownership["card_group"].eq("virtual_only"),
        "cliente_id",
    ]
) & valid_customer_ids

print(f"Training customers: {len(train_customer_ids):,}")
print(f"Candidate customers: {len(candidate_customer_ids):,}")

Training customers: 4,894
Candidate customers: 6,148


In [64]:
# Target de activación
# Marca activación observada dentro de los 30 días posteriores a la emisión.

physical_cards = cards[
    cards["tipo"].eq("fisica")
].copy()

training_population = physical_cards[
    physical_cards["cliente_id"].isin(train_customer_ids)
].copy()

training_population["activation_delay_days"] = (
    training_population["fecha_activacion"]
    - training_population["fecha_emision"]
).dt.days

training_population["target"] = (
    training_population["activation_delay_days"]
    .between(0, ACTIVATION_WINDOW_DAYS)
    .fillna(False)
    .astype(int)
)

In [65]:
# Validez temporal del target
# Excluye registros sin fecha de emisión porque no permiten definir la ventana de activación.

training_population = training_population[
    training_population["fecha_emision"].notna()
].copy()

print(f"Training rows: {len(training_population):,}")

display(
    training_population["target"]
    .value_counts(dropna=False)
    .rename("customers")
    .to_frame()
)

print(
    f"Activation rate: "
    f"{training_population['target'].mean():.2%}"
)

Training rows: 4,893


,customers
target,
0,2579
1,2314


Activation rate: 47.29%


In [66]:
# Validación del target
# Confirma que no existan activaciones fuera de la secuencia temporal esperada.

print(
    "Activations before issue:",
    (training_population["activation_delay_days"] < 0).sum(),
)

print(
    "Activations after target window:",
    (
        training_population["activation_delay_days"]
        > ACTIVATION_WINDOW_DAYS
    ).sum(),
)

Activations before issue: 0
Activations after target window: 0


#### Observaciones

- La población de entrenamiento está compuesta por clientes `physical_and_virtual`, ya que representan clientes que previamente contaban con tarjeta virtual y posteriormente recibieron una tarjeta física.

- De los 4,894 clientes identificados en este grupo, 4,893 disponen de una fecha de emisión válida y pueden utilizarse para definir correctamente la ventana de activación.

- El target presenta 2,314 clientes con activación dentro de los 30 días posteriores a la emisión (47.29%) y 2,579 sin activación observada dentro de dicha ventana (52.71%).

- No se identificaron activaciones anteriores a la emisión ni posteriores a los 30 días definidos para el target, lo que confirma consistencia con el comportamiento temporal observado previamente.

- Se identificaron 6,148 clientes `virtual_only` presentes en el maestro de clientes como universo preliminar de candidatos. La elegibilidad final se determinará considerando que el cliente y su tarjeta virtual ya existieran en la fecha de referencia utilizada para el scoring.

## 2. Construcción de Features

In [67]:
# Cobertura mensual de clientes transaccionales
# Verifica si la caída de transacciones también refleja menor cobertura de clientes.

monthly_coverage = (
    transactions[
        transactions["fecha"].dt.year.eq(2026)
    ]
    .assign(month=transactions["fecha"].dt.to_period("M"))
    .groupby("month")
    .agg(
        transactions=("transaccion_id", "count"),
        unique_customers=("cliente_id", "nunique"),
    )
)

display(monthly_coverage)

,transactions,unique_customers
month,,
2026-01,6716,4244
2026-02,6449,4121
2026-03,7608,4577
2026-04,8368,4668
2026-05,16323,6204
2026-06,25318,6583
2026-07,910,553
2026-08,112,112
2026-09,107,106


In [68]:
# Fechas de referencia y elegibilidad de candidatos
# Define el snapshot y conserva únicamente clientes elegibles en esa fecha.

training_base = (
    training_population[
        ["cliente_id", "fecha_emision", "target"]
    ]
    .rename(columns={"fecha_emision": "reference_date"})
    .copy()
)

SCORING_DATE = pd.Timestamp(CONTROL_INDEX_DATE)

candidate_base = pd.DataFrame({
    "cliente_id": sorted(candidate_customer_ids),
    "reference_date": SCORING_DATE,
})

candidate_temporal = (
    candidate_base
    .merge(
        customers[
            ["cliente_id", "fecha_registro"]
        ],
        on="cliente_id",
        how="left",
    )
)

virtual_cards = (
    cards[
        cards["tipo"].eq("virtual")
        & cards["cliente_id"].isin(candidate_customer_ids)
    ]
    .groupby("cliente_id", as_index=False)["fecha_emision"]
    .min()
    .rename(columns={"fecha_emision": "virtual_issue_date"})
)

candidate_temporal = candidate_temporal.merge(
    virtual_cards,
    on="cliente_id",
    how="left",
)

print(
    "Customers registered after scoring date:",
    (candidate_temporal["fecha_registro"] > candidate_temporal["reference_date"]).sum(),
)

print(
    "Virtual cards issued after scoring date:",
    (candidate_temporal["virtual_issue_date"] > candidate_temporal["reference_date"]).sum(),
)

print(
    "Missing virtual issue date:",
    candidate_temporal["virtual_issue_date"].isna().sum(),
)

candidate_base = (
    candidate_temporal[
        (candidate_temporal["fecha_registro"] <= candidate_temporal["reference_date"])
        & (candidate_temporal["virtual_issue_date"] <= candidate_temporal["reference_date"])
        & candidate_temporal["virtual_issue_date"].notna()
    ]
    [["cliente_id", "reference_date"]]
    .copy()
)

print(f"Training rows: {len(training_base):,}")
print(f"Eligible candidates: {len(candidate_base):,}")
print(f"Scoring date: {SCORING_DATE}")

Customers registered after scoring date: 225
Virtual cards issued after scoring date: 195
Missing virtual issue date: 2
Training rows: 4,893
Eligible candidates: 5,920
Scoring date: 2026-05-01 00:00:00


In [69]:
# Elegibilidad temporal del entrenamiento
# Verifica que el cliente y su tarjeta virtual existieran antes de la emisión física.

train_virtual_issue = (
    cards[
        cards["tipo"].eq("virtual")
        & cards["cliente_id"].isin(training_base["cliente_id"])
    ]
    .groupby("cliente_id", as_index=False)["fecha_emision"]
    .min()
    .rename(columns={"fecha_emision": "virtual_issue_date"})
)

training_temporal = (
    training_base
    .merge(
        train_virtual_issue,
        on="cliente_id",
        how="left",
    )
    .merge(
        customers[
            ["cliente_id", "fecha_registro"]
        ],
        on="cliente_id",
        how="left",
    )
)

print(
    "Missing virtual issue date:",
    training_temporal["virtual_issue_date"].isna().sum(),
)

print(
    "Virtual issued after physical:",
    (
        training_temporal["virtual_issue_date"]
        > training_temporal["reference_date"]
    ).sum(),
)

print(
    "Virtual issued same day as physical:",
    (
        training_temporal["virtual_issue_date"]
        == training_temporal["reference_date"]
    ).sum(),
)

print(
    "Customer registered after physical:",
    (
        training_temporal["fecha_registro"]
        > training_temporal["reference_date"]
    ).sum(),
)

training_base = (
    training_temporal[
        training_temporal["virtual_issue_date"].notna()
        & (
            training_temporal["virtual_issue_date"]
            <= training_temporal["reference_date"]
        )
        & (
            training_temporal["fecha_registro"]
            <= training_temporal["reference_date"]
        )
    ]
    [["cliente_id", "reference_date", "target"]]
    .copy()
)

print(f"Final training rows: {len(training_base):,}")

Missing virtual issue date: 5
Virtual issued after physical: 4
Virtual issued same day as physical: 4
Customer registered after physical: 25
Final training rows: 4,860


In [70]:
# Features transaccionales
# Resume la actividad durante los 60 días previos a la fecha de referencia.

def build_transaction_features(
    base: pd.DataFrame,
    transactions: pd.DataFrame,
    window_days: int,
) -> pd.DataFrame:

    tx = (
        transactions[
            transactions["cliente_id"].isin(base["cliente_id"])
        ]
        .merge(
            base[["cliente_id", "reference_date"]],
            on="cliente_id",
            how="inner",
        )
        .copy()
    )

    tx["days_from_reference"] = (
        tx["fecha"] - tx["reference_date"]
    ).dt.days

    tx = tx[
        tx["days_from_reference"].between(-window_days, -1)
    ].copy()

    tx["amount_volume"] = (
        pd.to_numeric(tx["monto"], errors="coerce").abs()
    )

    features = (
        tx
        .groupby("cliente_id", as_index=False)
        .agg(
            tx_count=("transaccion_id", "count"),
            total_amount=("amount_volume", "sum"),
            active_days=("fecha", "nunique"),
            avg_amount=("amount_volume", "mean"),
            last_transaction_date=("fecha", "max"),
        )
    )

    features = base.merge(
        features,
        on="cliente_id",
        how="left",
    )

    features["recency_days"] = (
        features["reference_date"]
        - features["last_transaction_date"]
    ).dt.days

    features[
        ["tx_count", "total_amount", "active_days", "avg_amount"]
    ] = features[
        ["tx_count", "total_amount", "active_days", "avg_amount"]
    ].fillna(0)

    features["recency_days"] = (
        features["recency_days"]
        .fillna(window_days + 1)
    )

    return features.drop(columns=["last_transaction_date"])

In [71]:
# Construcción y validación de features
# Aplica la misma lógica a entrenamiento y candidatos y compara sus distribuciones.

train_features = build_transaction_features(
    training_base,
    transactions,
    PRE_WINDOW_DAYS,
)

candidate_features = build_transaction_features(
    candidate_base,
    transactions,
    PRE_WINDOW_DAYS,
)

print(f"Train features: {train_features.shape}")
print(f"Candidate features: {candidate_features.shape}")

transaction_feature_cols = [
    "tx_count",
    "total_amount",
    "active_days",
    "avg_amount",
    "recency_days",
]

print("\nTraining")
display(
    train_features[transaction_feature_cols]
    .describe()
    .round(2)
)

print("\nCandidates")
display(
    candidate_features[transaction_feature_cols]
    .describe()
    .round(2)
)

Train features: (4860, 8)
Candidate features: (5920, 7)

Training


,tx_count,total_amount,active_days,avg_amount,recency_days
count,4860.00,4860.00,4860.00,4860.00,4860.00
mean,1.82,54.45,1.65,17.58,37.96
std,3.33,101.92,2.66,19.53,23.41
min,0.00,0.00,0.00,0.00,1.00
25%,0.00,0.00,0.00,0.00,14.00
50%,1.00,16.74,1.00,14.42,43.00
75%,2.00,62.30,2.00,30.21,61.00
max,32.00,948.19,20.00,141.74,61.00



Candidates


,tx_count,total_amount,active_days,avg_amount,recency_days
count,5920.00,5920.00,5920.00,5920.00,5920.00
mean,1.38,41.39,1.34,17.41,38.50
std,2.03,65.81,1.91,19.61,23.03
min,0.00,0.00,0.00,0.00,1.00
25%,0.00,0.00,0.00,0.00,15.00
50%,1.00,15.62,1.00,13.33,45.00
75%,2.00,56.02,2.00,30.17,61.00
max,20.00,760.82,17.00,148.45,61.00


In [72]:
# Features de cliente
# Deriva edad y antigüedad utilizando únicamente información válida al momento de referencia.

def add_customer_features(
    features: pd.DataFrame,
    customers: pd.DataFrame,
) -> pd.DataFrame:

    customer_data = customers[
        [
            "cliente_id",
            "fecha_nacimiento",
            "ciudad",
            "canal_adquisicion",
            "fecha_registro",
        ]
    ].copy()

    customer_data["fecha_nacimiento"] = pd.to_datetime(
        customer_data["fecha_nacimiento"],
        errors="coerce",
    )

    result = features.merge(
        customer_data,
        on="cliente_id",
        how="left",
    )

    # Fechas de nacimiento no plausibles
    invalid_birth = (
        result["fecha_nacimiento"].lt(pd.Timestamp("1900-01-01"))
        | result["fecha_nacimiento"].gt(result["reference_date"])
    )

    result.loc[
        invalid_birth,
        "fecha_nacimiento",
    ] = pd.NaT

    result["age"] = (
        (
            result["reference_date"]
            - result["fecha_nacimiento"]
        ).dt.days
        / 365.25
    )

    result["tenure_days"] = (
        result["reference_date"]
        - result["fecha_registro"]
    ).dt.days

    result["canal_adquisicion"] = (
        result["canal_adquisicion"]
        .fillna("desconocido")
    )

    return result.drop(
        columns=[
            "fecha_nacimiento",
            "fecha_registro",
        ]
    )

train_features = add_customer_features(
    train_features,
    customers,
)

candidate_features = add_customer_features(
    candidate_features,
    customers,
)

print(f"Train final features: {train_features.shape}")
print(f"Candidate final features: {candidate_features.shape}")

Train final features: (4860, 12)
Candidate final features: (5920, 11)


In [73]:
# Validación de features de cliente
# Revisa rangos y faltantes después del tratamiento de fechas inválidas.

print("Training")

display(
    train_features[
        ["age", "tenure_days"]
    ]
    .describe()
    .round(2)
)

print("Missing age:", train_features["age"].isna().sum())
print("Negative tenure:", (train_features["tenure_days"] < 0).sum())

print("\nCandidates")

display(
    candidate_features[
        ["age", "tenure_days"]
    ]
    .describe()
    .round(2)
)

print("Missing age:", candidate_features["age"].isna().sum())
print("Negative tenure:", (candidate_features["tenure_days"] < 0).sum())

Training


,age,tenure_days
count,4816.00,4860.00
mean,46.97,952.36
std,17.32,576.39
min,1.57,1.00
25%,32.08,451.00
50%,46.51,942.00
75%,61.46,1460.00
max,109.65,1946.00


Missing age: 44
Negative tenure: 0

Candidates


,age,tenure_days
count,5855.00,5920.00
mean,47.06,980.07
std,17.13,561.55
min,17.84,0.00
25%,32.36,503.00
50%,46.64,980.50
75%,61.74,1468.00
max,109.60,1946.00


Missing age: 65
Negative tenure: 0


In [74]:
# Revisión de edades extremas
# Cuantifica edades poco plausibles antes del modelado.

print(
    "Training age < 18:",
    (train_features["age"] < 18).sum(),
)

print(
    "Training age > 100:",
    (train_features["age"] > 100).sum(),
)

print(
    "Candidates age < 18:",
    (candidate_features["age"] < 18).sum(),
)

print(
    "Candidates age > 100:",
    (candidate_features["age"] > 100).sum(),
)

Training age < 18: 12
Training age > 100: 23
Candidates age < 18: 16
Candidates age > 100: 30


#### Observaciones

- Las features transaccionales se construyeron utilizando una ventana de 60 días previa a la fecha de referencia de cada cliente. Para entrenamiento se utilizó la `fecha_emision` de la tarjeta física y para candidatos el 1 de mayo de 2026.

- Inicialmente se consideró utilizar una fecha de scoring posterior al piloto, cercana a julio de 2026. Sin embargo, al revisar la cobertura transaccional se identificó una caída abrupta desde julio: los clientes con actividad pasan de 6,583 en junio a 553 en julio y cerca de 100 en agosto y septiembre. Esto indica una pérdida de cobertura del dataset y no permite utilizar de forma confiable los meses más recientes. Entre las fechas con información suficiente, se seleccionó el 1 de mayo de 2026 porque además coincide con la mediana de emisión del grupo tratado, mejorando la comparabilidad temporal entre entrenamiento y candidatos.

- La elegibilidad se validó temporalmente antes de construir las features. El conjunto final de entrenamiento conserva 4,860 clientes cuya tarjeta virtual y registro de cliente existían como máximo al momento de emisión de la tarjeta física. Para scoring quedaron 5,920 clientes `virtual_only` que ya existían y disponían de tarjeta virtual al momento del snapshot.

- Las variables transaccionales utilizadas son frecuencia (`tx_count`), volumen transaccionado (`total_amount`), días activos (`active_days`), ticket promedio (`avg_amount`) y recencia (`recency_days`). El volumen se calcula utilizando la magnitud absoluta de `monto`, manteniendo el criterio definido en el análisis del piloto.

- Las distribuciones transaccionales de entrenamiento y candidatos son razonablemente similares, aunque el grupo de entrenamiento presenta una actividad ligeramente superior. No se observa un corrimiento extremo entre ambas poblaciones en las variables construidas.

- Del maestro de clientes se incorporan edad, antigüedad, ciudad y canal de adquisición. Identificadores como cédula y nombre no se utilizan como features, y `estado_cuenta` se excluye por corresponder a un estado potencialmente dinámico cuyo momento de observación no está documentado.

- Las fechas de nacimiento claramente inconsistentes se trataron como faltantes antes de derivar la edad. Debido a la naturaleza sintética del dataset, las edades extremas técnicamente válidas se conservaron para evitar imponer reglas externas no especificadas en el ejercicio.

- La misma lógica de construcción de features se aplica tanto a entrenamiento como a candidatos, reduciendo el riesgo de inconsistencias entre entrenamiento y scoring.

## 3. Entrenamiento y Evaluación

In [75]:
# Preparación para entrenamiento
# Define variables predictoras y divide la muestra preservando la proporción del target.

from sklearn.model_selection import train_test_split

numeric_features = [
    "tx_count",
    "total_amount",
    "active_days",
    "avg_amount",
    "recency_days",
    "age",
    "tenure_days",
]

categorical_features = [
    "ciudad",
    "canal_adquisicion",
]

feature_cols = numeric_features + categorical_features

X = train_features[feature_cols].copy()
y = train_features["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Train: {len(X_train):,}")
print(f"Test: {len(X_test):,}")

print(f"\nTarget train: {y_train.mean():.2%}")
print(f"Target test: {y_test.mean():.2%}")

Train: 3,888
Test: 972

Target train: 47.22%
Target test: 47.22%


In [76]:
# Baseline por actividad previa
# Usa la frecuencia transaccional como criterio simple de priorización.

baseline_scores = X_test["tx_count"].copy()

baseline_results = pd.DataFrame({
    "score": baseline_scores,
    "target": y_test.values,
})

baseline_results["decile"] = pd.qcut(
    baseline_results["score"].rank(method="first"),
    q=10,
    labels=False,
) + 1

baseline_lift = (
    baseline_results
    .groupby("decile")
    .agg(
        customers=("target", "size"),
        activation_rate=("target", "mean"),
    )
    .sort_index(ascending=False)
)

overall_activation_rate = y_test.mean()

baseline_lift["lift"] = (
    baseline_lift["activation_rate"]
    / overall_activation_rate
)

display(baseline_lift.round(3))

,customers,activation_rate,lift
decile,,,
10,98,0.541,1.145
9,97,0.619,1.310
8,97,0.536,1.135
7,97,0.454,0.961
6,97,0.474,1.004
5,97,0.392,0.830
4,97,0.402,0.851
3,97,0.485,1.026
2,97,0.371,0.786


In [77]:
# Top decile baseline
# Resume el desempeño del 10% priorizado por actividad previa.

top_decile_baseline = baseline_lift.iloc[0]

print(
    f"Baseline top-decile activation rate: "
    f"{top_decile_baseline['activation_rate']:.2%}"
)

print(
    f"Baseline top-decile lift: "
    f"{top_decile_baseline['lift']:.2f}x"
)

Baseline top-decile activation rate: 54.08%
Baseline top-decile lift: 1.15x


In [78]:
# Pipeline de regresión logística
# Imputa faltantes, escala numéricas y codifica categóricas antes de entrenar.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

logistic_pipeline.fit(
    X_train,
    y_train,
)

logistic_scores = logistic_pipeline.predict_proba(
    X_test
)[:, 1]

logistic_auc = roc_auc_score(
    y_test,
    logistic_scores,
)

print(f"Logistic Regression ROC-AUC: {logistic_auc:.3f}")

Logistic Regression ROC-AUC: 0.629


In [79]:
# Lift de la regresión logística
# Evalúa si el modelo concentra activaciones en los clientes con mayor score.

logistic_results = pd.DataFrame({
    "score": logistic_scores,
    "target": y_test.values,
})

logistic_results["decile"] = pd.qcut(
    logistic_results["score"].rank(method="first"),
    q=10,
    labels=False,
) + 1

logistic_lift = (
    logistic_results
    .groupby("decile")
    .agg(
        customers=("target", "size"),
        activation_rate=("target", "mean"),
    )
    .sort_index(ascending=False)
)

overall_activation_rate = y_test.mean()

logistic_lift["lift"] = (
    logistic_lift["activation_rate"]
    / overall_activation_rate
)

display(logistic_lift.round(3))

,customers,activation_rate,lift
decile,,,
10,98,0.684,1.448
9,97,0.588,1.244
8,97,0.536,1.135
7,97,0.505,1.070
6,97,0.454,0.961
5,97,0.474,1.004
4,97,0.485,1.026
3,97,0.402,0.851
2,97,0.309,0.655


In [80]:
# Top decile del modelo
# Resume el desempeño del 10% con mayor probabilidad estimada.

top_decile_logistic = logistic_lift.iloc[0]

print(
    f"Logistic top-decile activation rate: "
    f"{top_decile_logistic['activation_rate']:.2%}"
)

print(
    f"Logistic top-decile lift: "
    f"{top_decile_logistic['lift']:.2f}x"
)

Logistic top-decile activation rate: 68.37%
Logistic top-decile lift: 1.45x


In [81]:
# Comparación contra baseline
# Resume la mejora obtenida por el modelo en el segmento de mayor prioridad.

model_comparison = pd.DataFrame({
    "approach": [
        "Baseline - tx_count",
        "Logistic Regression",
    ],
    "top_10_activation_rate": [
        top_decile_baseline["activation_rate"],
        top_decile_logistic["activation_rate"],
    ],
    "top_10_lift": [
        top_decile_baseline["lift"],
        top_decile_logistic["lift"],
    ],
})

display(model_comparison.round(3))

,approach,top_10_activation_rate,top_10_lift
0,Baseline - tx_count,0.541,1.145
1,Logistic Regression,0.684,1.448


In [82]:
# Evaluación por cobertura
# Mide el desempeño al priorizar distintos porcentajes de clientes.

ranked_test = pd.DataFrame({
    "score": logistic_scores,
    "target": y_test.values,
}).sort_values(
    "score",
    ascending=False,
).reset_index(drop=True)

coverage_results = []

for coverage in [0.10, 0.20, 0.30, 0.40, 0.50]:
    n_selected = int(np.ceil(len(ranked_test) * coverage))

    selected = ranked_test.head(n_selected)

    activation_rate = selected["target"].mean()
    lift = activation_rate / overall_activation_rate
    captured_activations = (
        selected["target"].sum()
        / ranked_test["target"].sum()
    )

    coverage_results.append({
        "coverage": coverage,
        "customers_selected": n_selected,
        "activation_rate": activation_rate,
        "lift": lift,
        "captured_activations": captured_activations,
    })

coverage_table = pd.DataFrame(coverage_results)

display(
    coverage_table.style.format({
        "coverage": "{:.0%}",
        "activation_rate": "{:.2%}",
        "lift": "{:.2f}x",
        "captured_activations": "{:.2%}",
    })
)

,coverage,customers_selected,activation_rate,lift,captured_activations
0,10%,98,68.37%,1.45x,14.60%
1,20%,195,63.59%,1.35x,27.02%
2,30%,292,60.27%,1.28x,38.34%
3,40%,389,57.84%,1.22x,49.02%
4,50%,486,55.35%,1.17x,58.61%


In [83]:
# Importancia de variables
# Extrae los coeficientes del modelo para interpretar las principales señales.

feature_names = (
    logistic_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients = (
    logistic_pipeline
    .named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

coefficient_table["feature"] = (
    coefficient_table["feature"]
    .str.replace("numeric__", "", regex=False)
    .str.replace("categorical__", "", regex=False)
)

coefficient_table["abs_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values(
        "abs_coefficient",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    coefficient_table[
        ["feature", "coefficient"]
    ]
    .head(15)
    .round(3)
)

,feature,coefficient
0,canal_adquisicion_referido,0.691
1,canal_adquisicion_call_center,-0.298
2,ciudad_quito,0.273
3,active_days,0.271
4,age,-0.269
5,ciudad_ambato,0.231
6,canal_adquisicion_desconocido,-0.230
7,ciudad_loja,-0.181
8,tenure_days,0.167
9,canal_adquisicion_publicidad_digital,-0.156


In [84]:
# Señales positivas y negativas
# Separa las variables asociadas con mayor y menor probabilidad estimada de activación.

positive_drivers = (
    coefficient_table[
        coefficient_table["coefficient"] > 0
    ]
    .sort_values("coefficient", ascending=False)
    .head(8)
)

negative_drivers = (
    coefficient_table[
        coefficient_table["coefficient"] < 0
    ]
    .sort_values("coefficient")
    .head(8)
)

print("Positive drivers")
display(
    positive_drivers[
        ["feature", "coefficient"]
    ].round(3)
)

print("Negative drivers")
display(
    negative_drivers[
        ["feature", "coefficient"]
    ].round(3)
)

Positive drivers


,feature,coefficient
0,canal_adquisicion_referido,0.691
2,ciudad_quito,0.273
3,active_days,0.271
5,ciudad_ambato,0.231
8,tenure_days,0.167
12,avg_amount,0.106
19,tx_count,0.040


Negative drivers


,feature,coefficient
1,canal_adquisicion_call_center,-0.298
4,age,-0.269
6,canal_adquisicion_desconocido,-0.230
7,ciudad_loja,-0.181
9,canal_adquisicion_publicidad_digital,-0.156
10,ciudad_machala,-0.156
11,canal_adquisicion_organico,-0.139
13,total_amount,-0.098


#### Resultado del modelo

| Indicador | Resultado | Lectura |
|---|---:|---|
| **ROC-AUC** | **0.629** | Capacidad de discriminación moderada |
| **Top 10% - activación** | **68.37%** | Frente a 47.22% de activación general |
| **Top 10% - lift** | **1.45x** | 45% mejor que selección aleatoria |
| **Baseline top 10%** | **54.08% / 1.15x** | El modelo supera la regla basada solo en `tx_count` |
| **Top 20% - activación** | **63.59%** | Mantiene buen desempeño con mayor cobertura |
| **Top 30% - activación** | **60.27%** | El ranking sigue concentrando activadores |

<br>

> **Lectura de negocio:** El modelo busca ordenar mejor a quién priorizar cuando el número de tarjetas es limitado. En el top 10% del ranking, la tasa de activación observada sube de 47.22% a 68.37%.

<br>

> **Decisión:** Se selecciona la regresión logística como modelo final por superar el baseline, mantener interpretabilidad y generar un ranking útil para distintos niveles de presupuesto.

#### Observaciones

- El ranking generado por la regresión logística concentra de forma progresiva una mayor proporción de activadores en los niveles superiores, por lo que resulta adecuado para un escenario donde la disponibilidad de tarjetas es limitada.

- Entre las señales con mayor asociación positiva destacan el canal de adquisición `referido`, los días activos previos y la antigüedad del cliente. Entre las asociaciones negativas aparecen `call_center`, edad y determinados canales o ciudades.

- Los coeficientes representan asociaciones dentro del modelo y no relaciones causales. En particular, variables transaccionales correlacionadas como volumen total, frecuencia y ticket promedio deben interpretarse conjuntamente.

- Se prioriza la regresión logística por combinar mejora frente al baseline, utilidad para ranking e interpretabilidad para negocio, sin añadir complejidad innecesaria.

- No se realizó una optimización de hiperparámetros exhaustiva. Aunque podría aplicarse un `GridSearchCV` para ajustar parámetros como `C`, la regresión logística sin optimización ya mostró una mejora clara frente al baseline y un lift relevante en los segmentos priorizados. Dado el alcance práctico del ejercicio, se priorizó mantener una solución simple, interpretable y suficientemente efectiva antes que incrementar complejidad con una ganancia potencialmente marginal.

## 4. Priorización de la Siguiente Ola

In [88]:
# Modelo final
# Reentrena la solución seleccionada utilizando toda la población disponible.

from sklearn.base import clone

final_pipeline = clone(logistic_pipeline)

final_pipeline.fit(
    X,
    y,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['tx_count','total_amount','active_days',...,'tenure_days','ciudad', 'canal_adquisicion']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``rem

In [90]:
# Scoring y ranking de candidatos
# Ordena clientes según su probabilidad estimada de activación.

candidate_scores = final_pipeline.predict_proba(
    candidate_features[feature_cols]
)[:, 1]

candidate_ranking = candidate_features[
    ["cliente_id"]
].copy()

candidate_ranking["priority_score"] = candidate_scores

candidate_ranking["priority_rank"] = (
    candidate_ranking["priority_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

candidate_ranking["priority_decile"] = pd.qcut(
    candidate_ranking["priority_rank"],
    q=10,
    labels=list(range(10, 0, -1)),
).astype(int)

candidate_ranking = (
    candidate_ranking
    .sort_values("priority_rank")
    .reset_index(drop=True)
)

display(candidate_ranking.head(10))

,cliente_id,priority_score,priority_rank,priority_decile
0,CHK-006247,0.890416,1,10
1,CHK-002874,0.865100,2,10
2,CHK-001776,0.863112,3,10
3,CHK-006503,0.846077,4,10
4,CHK-009818,0.845004,5,10
5,CHK-003273,0.842872,6,10
6,CHK-008222,0.839484,7,10
7,CHK-011673,0.836864,8,10
8,CHK-007157,0.836349,9,10
9,CHK-006926,0.828527,10,10


In [87]:
# Validación del ranking
# Revisa distribución de scores y tamaño de los deciles.

display(
    candidate_ranking["priority_score"]
    .describe()
    .to_frame()
    .round(3)
)

display(
    candidate_ranking["priority_decile"]
    .value_counts()
    .sort_index(ascending=False)
    .to_frame("customers")
)

,priority_score
count,5920.000
mean,0.469
std,0.125
min,0.178
25%,0.375
50%,0.456
75%,0.551
max,0.892


,customers
priority_decile,
10,592
9,592
8,592
7,592
6,592
5,592
4,592
3,592
2,592


#### Observaciones

- El modelo fue aplicado sobre los 5,920 clientes `virtual_only` elegibles, generando para cada uno una probabilidad estimada de activación (`priority_score`) y un ranking de prioridad.

- Los scores se distribuyen entre 0.178 y 0.892, con una mediana de 0.456, permitiendo diferenciar distintos niveles de prioridad dentro de la población candidata.

- El ranking se dividió en diez deciles de 592 clientes cada uno. El decil 10 concentra el 10% de clientes con mayor score estimado y representa el grupo de mayor prioridad.

- No se define un número fijo de clientes recomendados debido a que el ejercicio no especifica cuántas tarjetas estarán disponibles. En su lugar, el ranking permite adaptar la selección al presupuesto real de negocio utilizando `priority_rank` o `priority_decile`.

- La priorización representa probabilidad estimada de activación bajo el comportamiento observado en el piloto y debe interpretarse como criterio de asignación, no como garantía individual de activación.